# 01 — Method A: **BF16 LoRA** 베이스라인 (KorQuAD)

발표(Serve Track: LoRA → INT4 PTQ/QAT → vLLM)용 **3-way 양자화 비교**의 기준선.

| 방법 | 무엇 | 언제 |
|---|---|---|
| **A = BF16 LoRA** (이 노트북) | 풀정밀(BF16) 베이스에 LoRA 어댑터 학습 후 **머지** | 품질 상한 기준선 |
| B = INT4 PTQ (02) | A의 머지 모델을 **사후** 4bit 양자화 | 학습 없이 압축 |
| C = INT4 QAT (03) | 4bit **인식 학습** | 양자화 오차를 학습으로 보정 |

A는 B·C의 **입력(머지 BF16 모델)**이자 품질 기준이므로 먼저 확정합니다.

> ## ⚙️ 실행 모드 배너 — 이 노트북은 **Azure A100 80GB에서 실제 실행**됨 (프로덕션/스펙 경로)
>
> **컴퓨트:** Azure `Standard_NC24ads_A100_v4` (**NVIDIA A100 80GB PCIe** ×1) · region **japaneast** ·
> `compute.mode: gpu`. 모던 GPU 쿼터는 여러 리전 분산 신청으로 확보했고(japaneast A100 96 vCPU),
> MCAPS 거버넌스 정책이 온디맨드 GPU SKU 배포를 차단하므로 **Spot 우선순위**로 프로비저닝했습니다.
>
> **설정(스펙 A):** base=`Qwen/Qwen3-1.7B`, `train.backend: unsloth`, **BF16**(load_in_4bit=false),
> LoRA r=16/α=32(attn+MLP), KorQuAD **max_steps=2000** (eff.batch 8 ≈ 16k examples, ~0.26 epoch), held-out eval 500.
>
> **재현:** 아래 셀들은 CPU 스모크와 **동일 코드**이며 설정만 다릅니다. 버전 고정값은
> `results/env_A.json`, 학습 로그는 `results/A_train_log.json`, 수치는 `results/A_bf16_metrics.json`에 기록됩니다.


### 0) 부트스트랩 & 버전 고정 (재현성)
`quantization/` 공용 모듈을 import하고, 정확한 버전을 `results/env_A.json`에 기록.

In [1]:
import os, sys
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        os.chdir(cand)
        break
print("cwd:", os.getcwd())

cwd: /home/azureuser/work/pdf_qa_extraction


In [2]:
import json, platform, torch, transformers, trl, peft, datasets
env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "peft": peft.__version__,
    "datasets": datasets.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
}
os.makedirs("quantization/results", exist_ok=True)
with open("quantization/results/env_A.json", "w", encoding="utf-8") as fh:
    json.dump(env, fh, ensure_ascii=False, indent=2)
env

{'python': '3.10.12',
 'torch': '2.11.0+cu130',
 'transformers': '5.5.0',
 'trl': '0.24.0',
 'peft': '0.20.0',
 'datasets': '4.3.0',
 'cuda_available': True,
 'device': 'NVIDIA A100 80GB PCIe'}

### 1) 설정 로드
`compute.mode: cpu` 스모크 오버라이드가 적용됩니다(소형 모델·서브셋·12스텝). GPU VM에선 `mode: gpu`로 두면 됩니다.

In [3]:
from quantization.data_korquad import load_config, load_korquad, to_hf_text_dataset
cfg = load_config()   # reads compute.mode: gpu from config.yaml (A100 production run)
print('base :', cfg['base_model']['selected'])
print('mode :', cfg['compute']['mode'], '| backend:', cfg['train']['backend'],
      '| precision:', cfg['train']['precision'])
print('lora :', cfg['lora'])
print('data :', {k: cfg['data'][k] for k in ['dataset','seed','eval_size','train_subset','max_seq_len']})

base : Qwen/Qwen3-1.7B
mode : gpu | backend: unsloth | precision: bf16
lora : {'r': 16, 'alpha': 32, 'dropout': 0.0, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']}
data : {'dataset': 'KorQuAD/squad_kor_v1', 'seed': 42, 'eval_size': 500, 'train_subset': None, 'max_seq_len': 1024}


### 2) 데이터 — KorQuAD → 생성 instruction 포맷
런타임 다운로드(레포 미커밋). 고정 seed로 held-out val 슬라이스(A/B/C 동일).

In [4]:
data = load_korquad(cfg)
print('train:', len(data['train']), '| eval(held-out):', len(data['eval']))
ex = data['eval'][0]
print('\n--- prompt ---\n' + ex.prompt)
print('--- gold answers ---', ex.answers)

train: 60407 | eval(held-out): 500

--- prompt ---
아래 문맥을 읽고 질문에 답하세요.

[문맥]
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교통비 절감 효과로 이용객이 늘어나자 버스, 지하철 회사의 수입이 증가하였다. 초기엔 불편을 토로하던 시민들도 정착 후에는 바뀐 교통체계를 지지하는 사람들이 많아졌고 이는 이명박의 대중 인기 증가에 큰 보탬이 되었다. 이에 힘입어 이명박은 서울시장 퇴임 후 대선 후보에 올라 당선되기에 이른다.

[질문] 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[답]

--- gold answers --- ['대중교통체계']


### 3) EM/F1 지표 자가검증 (KorQuAD 공식 · 한국어 char-level)
SQuAD 나이브 EM이 아니라 KorQuAD 정규화 + 문자단위 F1임을 확인.

In [5]:
import quantization.eval_qa as E
E._selftest()
print('공식 정규화 예:', repr(E.normalize_answer('세종대왕! (조선)')))

[eval selftest] EM/F1 over 3 pairs: {'exact_match': 66.66666666666667, 'f1': 90.90909090909092, 'n': 3}
[eval selftest] OK
공식 정규화 예: '세종대왕 조선'


### 4) 학습 — BF16 LoRA (r=16, α=32, attn+MLP)
스모크: 0.5B·12스텝. 산출물 = LoRA 어댑터 + **머지 모델**(`artifacts/A_bf16/`, Part 2 입력).

In [6]:
from quantization.train_lora import train
train_log = train(cfg)
train_log

/home/azureuser/venv/lib/python3.10/site-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:24<00:00, 24.02s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:24<00:00, 24.02s/it]

Unsloth: Merge process complete. Saved to `/home/azureuser/work/pdf_qa_extraction/quantization/artifacts/A_bf16`


{'base_model': 'Qwen/Qwen3-1.7B',
 'backend': 'unsloth',
 'precision': 'bf16',
 'mode': 'gpu',
 'n_train': 60407,
 'train_seconds': 1986.18,
 'train_loss': 2.0293112831115723,
 'global_step': 2000,
 'adapter_dir': 'quantization/artifacts/A_bf16_adapter',
 'merged_dir': 'quantization/artifacts/A_bf16'}

### 5) 동작 데모 (필수) — held-out 질문 **1개**
튜닝된 머지 모델을 로드해 실제로 답을 생성합니다(“동작한다”를 눈으로).

In [7]:
model, tok = E.load_model_for_eval(cfg['paths']['method_a_dir'], cfg['train']['precision'])
demo = data['eval'][0]
gen = E.generate_answers(model, tok, [demo.prompt], max_new_tokens=32, batch_size=1)
print('[질문]', demo.question)
print('[정답]', demo.answers)
print('[모델 답]', gen['answers'][0])

==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

The tokenizer you are loading from 'quantization/artifacts/A_bf16' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


The tokenizer you are loading from 'quantization/artifacts/A_bf16' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


/home/azureuser/venv/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/azureuser/venv/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


[질문] 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[정답] ['대중교통체계']
[모델 답] 대중교통체계


### 6) 수치 — EM/F1 · perplexity · 크기 · VRAM · tok/s
`results/A_bf16_metrics.json` 저장 + 3-way 표 첫 행(`three_way_table.json`) append.

In [8]:
res = E.evaluate_model(model, tok, data['eval'], method='A_bf16',
                       base_model=cfg['base_model']['selected'],
                       model_dir=cfg['paths']['method_a_dir'],
                       max_new_tokens=cfg['eval']['max_new_tokens'],
                       batch_size=cfg['eval']['batch_size'],
                       ppl_samples=cfg['eval']['ppl_samples'],
                       precision=cfg['train']['precision'],
                       notes=f"mode={cfg['compute']['mode']} backend={cfg['train']['backend']}")
E.write_metrics(res, cfg['paths']['results_dir'])
E.append_to_table(res, cfg['paths']['results_dir'])
from dataclasses import asdict
row = asdict(res)
print('A (BF16 LoRA) — 3-way 표 첫 행')
for k in ['method','base_model','exact_match','f1','perplexity','size_gb','peak_vram_gb','tok_per_s','precision']:
    print(f'  {k:14}: {row[k]}')

Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


/home/azureuser/venv/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/azureuser/venv/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/azureuser/venv/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers

Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A (BF16 LoRA) — 3-way 표 첫 행
  method        : A_bf16
  base_model    : Qwen/Qwen3-1.7B
  exact_match   : 81.0
  f1            : 89.924
  perplexity    : 10.3928
  size_gb       : 3.2155
  peak_vram_gb  : 7.9251
  tok_per_s     : 122.75
  precision     : bf16


### 7) 다음 단계 (Part 2)
머지 BF16 모델(`artifacts/A_bf16/`)을 입력으로 **B. INT4 PTQ**(`02`)와 **C. INT4 QAT**(`03`)를 동일 템플릿·동일 eval로 실행해 3-way 표를 완성하고 vLLM 서빙 벤치로 잇습니다.

> 프로덕션 재현: GPU VM에서 `compute.mode: gpu`로 이 노트북을 재실행하면 표의 A 행이 Qwen3-1.7B BF16(A100/H100) 수치로 채워집니다.